In [1]:
import torch

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [3]:
from torchvision.transforms import v2
from torchvision.datasets import CIFAR10
from torch import nn

CIFAR_MEAN = (0.49145, 0.48219, 0.44658)
CIFAR_STD = (0.24687, 0.24332, 0.26137)

crop_transform = nn.ModuleList([
    v2.RandomCrop(
        size=32,
        padding=4
    )
])

train_transforms = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD)
])

test_transforms = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD)
])

train_dataset = CIFAR10(
    root="../data",
    train=True,
    transform=train_transforms,
    download=True
)

test_dataset = CIFAR10(
    root="../data",
    train=False,
    transform=test_transforms,
    download=True
)

categories = tuple(train_dataset.classes)

In [4]:
from torch.utils.data import DataLoader, random_split

# create validation set 
generator = torch.Generator().manual_seed(7)
train_dataset, val_dataset = random_split(train_dataset, [0.01, 0.99], generator=generator)

# create DataLoader objects
train_dl = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dl = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_dl = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [5]:
len(train_dataset), len(val_dataset), len(train_dl)

(500, 49500, 16)

In [6]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [7]:
from src.model import VisionTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

model = VisionTransformer().to(device, non_blocking=True)
model

VisionTransformer(
  (conv_layer): Conv2d(3, 512, kernel_size=(2, 2), stride=(2, 2))
  (dropout): Dropout(p=0.0, inplace=False)
  (transformer_encoder): TransformerEncoder(
    (transformer_layers): ModuleList(
      (0-7): 8 x TransformerLayer(
        (layer_norm_1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
        (layer_norm_2): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
        (multi_head_self_attention): MultiHeadSelfAttention(
          (self_attention_heads): ModuleList(
            (0-7): 8 x SelfAttentionHead(
              (Q_linear_transform): Linear(in_features=512, out_features=64, bias=False)
              (K_linear_transform): Linear(in_features=512, out_features=64, bias=False)
              (V_linear_transform): Linear(in_features=512, out_features=64, bias=False)
            )
          )
          (output_linear_transform): Linear(in_features=512, out_features=512, bias=False)
        )
        (mlp): MLP(
          (li

In [8]:
loss_fn = torch.nn.CrossEntropyLoss().to(device, non_blocking=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [9]:
model.train()
for epoch in range(1000):
    total_loss = 0
    for X_batch, y_batch in train_dl:
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)
    
        # 1. forward pass
        preds = model(X_batch)
    
        # 2. compute loss
        loss = loss_fn(preds, y_batch)
    
        # 3. reset gradients
        optimizer.zero_grad()
    
        # 4. compute gradients
        loss.backward()
    
        # 5. optimizer step
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch - {epoch}, Loss - {total_loss / len(train_dl)}")

Epoch - 0, Loss - 2.3001025170087814
Epoch - 1, Loss - 2.1320925801992416
Epoch - 2, Loss - 2.04757958650589
Epoch - 3, Loss - 2.0114822536706924
Epoch - 4, Loss - 1.9884494245052338
Epoch - 5, Loss - 1.930111974477768
Epoch - 6, Loss - 1.8761522099375725
Epoch - 7, Loss - 1.8083659783005714
Epoch - 8, Loss - 1.751315027475357
Epoch - 9, Loss - 1.6278745085000992
Epoch - 10, Loss - 1.4578580036759377
Epoch - 11, Loss - 1.230918899178505
Epoch - 12, Loss - 1.0111770406365395
Epoch - 13, Loss - 0.7874875478446484
Epoch - 14, Loss - 0.5863887593150139
Epoch - 15, Loss - 0.43032981269061565
Epoch - 16, Loss - 0.2835606886073947
Epoch - 17, Loss - 0.1421142164617777
Epoch - 18, Loss - 0.06860321969725192
Epoch - 19, Loss - 0.03694120980799198
Epoch - 20, Loss - 0.025202372577041388


KeyboardInterrupt: 

In [ ]:
from src.inference_functions import evaluate

evaluate(model, train_dl, device, loss_fn)